# scBIOT from an AnnData used by Seurat (v1.2.0)

The tracked `tutorials/R/ifnb.h5ad` object comes from SeuratData's IFNB study. CTRL cells provide labels and STIM cells are the query. The corresponding R Markdown tutorial shows the same calls through `reticulate`.

- Original source: [Seurat immune-alignment tutorial](https://satijalab.org/seurat/articles/integration_introduction)
- Reproducible local input: [`ifnb.h5ad`](https://github.com/haihuilab/scbiot/blob/main/tutorials/R/ifnb.h5ad)


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
path = ROOT / "R" / "ifnb.h5ad"
adata = sc.read_h5ad(path)
adata.obs["batch"] = adata.obs["stim"].astype(str)
adata.obs["cell_type"] = adata.obs["seurat_annotations"].astype(str)
adata.obs["semi_cell_type"] = np.where(
    adata.obs["stim"].astype(str).eq("CTRL"), adata.obs["cell_type"], "Unknown"
)
adata.layers["counts"] = adata.X.copy()
adata


## Embed, integrate, and transfer labels


In [ ]:
adata = scb.pp.autoencoder(
    adata,
    input_key="counts",
    out_key="X_ae",
    batch_key="batch",
    random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata,
    obsm_key="X_ae",
    batch_key="batch",
    out_key="X_supbiot",
    label_key="semi_cell_type",
    unlabeled_category="Unknown",
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
adata = scb.ot.supbiot(
    adata,
    use_rep="X_supbiot",
    input_rep_key="X_ae",
    label_key="semi_cell_type",
    unlabeled_category="Unknown",
    pred_label_key="pred_cell_type",
    pred_conf_key="pred_confidence",
    min_conf=0.0,
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


In [ ]:
from sklearn.metrics import normalized_mutual_info_score
mask = adata.obs["semi_cell_type"].astype(str).eq("Unknown")
score = normalized_mutual_info_score(
    adata.obs.loc[mask, "cell_type"].astype(str),
    adata.obs.loc[mask, "pred_cell_type"].astype(str),
)
print(f"Held-out NMI: {score:.3f}")
sc.pp.neighbors(adata, use_rep="X_supbiot", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["batch", "cell_type", "pred_cell_type"])
